In [5]:
import os
from pathlib import Path
from PIL import Image
import imagehash

# ─── CONFIG ────────────────────────────────────────────────────────────────────
# point this at the parent directory containing your *_clahe folders
BASE_DIR = Path(r"images_for_modeling\deforestation_drivers")

# which extensions to consider
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

# choose a hash function and size; phash is usually robust
HASH_FUNC = imagehash.phash
HASH_SIZE = 16

# max Hamming distance between hashes to consider "duplicates"
DUPLICATE_THRESHOLD = 65
# ────────────────────────────────────────────────────────────────────────────────

for folder in BASE_DIR.iterdir():
    if not (folder.is_dir() and folder.name.endswith("_clahe")):
        continue

    print(f"\nProcessing folder: {folder.name}")
    seen_hashes = []  # list of (hash, path)

    for img_path in folder.iterdir():
        if img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            img = Image.open(img_path)
            img_hash = HASH_FUNC(img, hash_size=HASH_SIZE)
        except Exception as e:
            print(f"  ⚠️  Skipping {img_path.name}: {e}")
            continue

        # compare against all previously seen hashes
        is_dup = False
        for prev_hash, prev_path in seen_hashes:
            if img_hash - prev_hash <= DUPLICATE_THRESHOLD:
                print(f"  🗑️  Removing duplicate {img_path.name} (matches {prev_path.name})")
                img_path.unlink()
                is_dup = True
                break

        if not is_dup:
            seen_hashes.append((img_hash, img_path))

print("\nDone.")



Processing folder: Oil palm plantation_clahe
  🗑️  Removing duplicate 2016_9.32209375_5.155937500000001.png (matches 2015_9.322625_5.155214285714286.png)
  🗑️  Removing duplicate 2017_10.666924999999997_3.6285249999999993.png (matches 2016_10.666250000000002_3.6279999999999997.png)
  🗑️  Removing duplicate 2017_9.435375_5.155125.png (matches 2016_9.435625_5.1545.png)
  🗑️  Removing duplicate 2019_10.237620654911247_3.9681255611340847.png (matches 2019_10.2374500989903_3.969013743813106.png)
  🗑️  Removing duplicate 2019_10.681239023071152_3.734208102864394.png (matches 2019_10.68087115121723_3.733167827743858.png)
  🗑️  Removing duplicate 2019_11.849776480400871_3.4216610023238774.png (matches 2019_11.848875000000001_3.421590001979806.png)
  🗑️  Removing duplicate 2019_9.892294482489135_3.859724674441107.png (matches 2019_9.89199078103346_3.8610248822015443.png)
  🗑️  Removing duplicate 2020_10.665575_3.628624999999999.png (matches 2016_10.666250000000002_3.6279999999999997.png)
  🗑️ 

In [2]:
!pip install imagehash


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
